# Chapter 8 — Linking Ontologies to Data
### Notebook 3 · Exercises

*Book reference: Section 8.4*

The book's exercises, executable. Assertions are the marking scheme.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch08_toolkit as ch8
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
conn = ch8.build_database()

### Exercise R1 — Explain where the ontology goes at run time

Show, for one query, the SPARQL, the SQL, and the fact that both give the same answer. Then state in one sentence what role the ontology played.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
q = ch8.QUERIES['patients-in-cardiology']
print('SPARQL over the materialised graph:'); print(ch8.to_sparql(q))
print('\nSQL over the original source:'); print(ch8.to_sql(q))
left = ch8.answers_via_materialisation(conn, q)
right = ch8.answers_via_rewriting(conn, q)
print('\nsame answers:', left == right, f'({len(left)} rows)')
assert left == right
print('\nUnder rewriting the ontology is a COMPILE-TIME artefact: it decided\n'
      'which tables to join and which columns to compare, then vanished. No\n'
      'ontology, no triples and no reasoner exist while the query runs.')

### Exercise R2 — Decide the strategy for three workloads

For each workload, choose materialise or rewrite and justify it in terms of freshness and query cost.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
workloads = [
    ('nightly report over a frozen snapshot, thousands of reads', 'materialise'),
    ('live clinical alerting over a transactional database', 'rewrite'),
    ('published reference data, never updated, heavily queried', 'materialise'),
]
for description, expected in workloads:
    volatile = any(w in description for w in ('live', 'transactional'))
    choice = 'rewrite' if volatile else 'materialise'
    print(f'{choice:12s} (expected {expected:12s}) <- {description}')
    assert choice == expected
print('\nThe question is never "which is better". It is: how often does the\n'
      'source change relative to how often it is queried, and what does a stale\n'
      'answer cost you?')

### Exercise R3 — Quantify the staleness window

Take a snapshot, apply a series of updates, and plot how wrong the stale copy becomes as updates accumulate.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
fresh_conn = ch8.build_database()
snapshot = ch8.materialise(fresh_conn)
q = ch8.QUERIES['patients-with-cardiac-disorder']
rows = []
for i in range(4):
    stale = ch8.answers_via_materialisation(fresh_conn, q, graph=snapshot)
    live = ch8.answers_via_rewriting(fresh_conn, q)
    missing = len(live) - len(stale)
    rows.append({'updates applied': i, 'stale answer': len(stale),
                 'true answer': len(live), 'rows missed': missing})
    pid = 200 + i
    fresh_conn.execute('INSERT INTO patient VALUES (?, ?, 1)', (pid, f'New{i}'))
    fresh_conn.execute("INSERT INTO diagnosis VALUES (?, 'I21')", (pid,))
    fresh_conn.commit()
import pandas as pd; print(pd.DataFrame(rows).to_string(index=False))
assert rows[-1]['rows missed'] == 3
print('\nError grows monotonically with time since refresh, and nothing in the\n'
      'system reports it. A refresh schedule is therefore a CORRECTNESS budget,\n'
      'not a maintenance chore.')

## Where this leaves you

You can map a schema to an ontology, run the result two ways, and detect the bug that neither path reports on its own. Notebook 4 turns the strategy choice into a decision problem where staleness is priced as lost reward.